# Метрики качества сети с цензурой: Sepsis и синтетика A–D

Ноутбук запускает `survpm.metrics.crossval` (библиотека лежит в `dima/survpm/`) на двух журналах
с 5-fold кросс-валидацией по трассам. В каждом фолде на 4/5 трасс оцениваются две модели
`CensoredStateNet`: **наивная MLE** (`naive`) и **MLE с учётом цензуры** (`censor_aware`).
Все метрики считаются на оставшейся 1/5. Фолды равны по доле цензурированных трасс.

| Уровень | Метрики | Библиотека |
|---|---|---|
| 1. Process mining | fitness и precision (token replay, alignments), generalization, simplicity; сеть состояний против Inductive Miner | PM4Py |
| 2. Вероятности путей | entropic relevance, uEMSC, EMD языков | Ebi; EMD — метрика PM4Py, решение через POT |
| 3. Время | temporal profile, CFLD, 3-граммы, EMD длительности трасс и времени событий, KS по рёбрам, расстояние до Каплана–Майера, log-rank, integrated Brier score | PM4Py, log-distance-measures, lifelines, scikit-survival |
| 4. Обобщение | −log L тестовых интервалов с цензурой, доля трасс с невиданными переходами | свой код |

**Варианты A и Б.** A — только полные тестовые трассы (дошли до терминала).
Б — все тестовые трассы, цензурированная трасса считается префиксом:
* уровень 1: token replay без финальной маркировки, оставшиеся фишки не штрафуются; precision (ETC) и так строится по префиксам; alignments только в A;
* уровень 2: в модель добавлена остановка в состоянии с наблюдаемой вероятностью цензуры `q`;
* уровень 3: симулированная трасса обрывается в момент цензуры парной тестовой трассы.
  Метрики выживания (`km_*`, `logrank_*`, `ibs`) учитывают цензуру сами и записаны в строку Б.

**Установка** (один раз):

```bash
pip install -e "survpm[metrics]"
# log-distance-measures формально требует Python < 3.13, но работает и на 3.13:
pip install --ignore-requires-python log-distance-measures jellyfish pulp
```

**Как читать.** Больше — лучше: fitness, precision, generalization, simplicity, uEMSC, `logrank_p`.
Меньше — лучше: entropic relevance (бит на трассу), EMD, CFLD, `ngram_3`, `cycle_time_emd` и
`absolute_event_emd` (часы), `edge_ks`, `km_sup`, `km_l1_days`, `ibs`, `temporal_deviation_share`, `nll_*`.


In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import pm4py
from IPython.display import display

dima_dir = Path.cwd() if Path("survpm").exists() else Path.cwd() / "dima"
sys.path.insert(0, str(dima_dir / "survpm" / "src"))  # не нужно, если survpm установлена через pip
from survpm import datasets
from survpm.metrics import crossval

pm4py.util.constants.SHOW_PROGRESS_BAR = False
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




In [2]:
# ===== Настройки =====
K_FOLDS = 5
SEED = 2026
N_SIM = 5000                 # трасс симуляции модели на фолд (уровни 2–3)
SEPSIS_PATH = dima_dir.parent / "Med_Eremina" / "Sepsis Cases - Event Log_grouped.csv"
SYNTHETIC_CASES = 1000
OUTPUT_DIR = dima_dir / "notebooks" / "output" / "censored_metrics"

## 1. Sepsis

Состояния — группы событий `event_group`, старт — первая `Registration`, терминал — первая `Discharge`.
Момент окончания наблюдения неизвестен, поэтому цензура наступает в момент последней записи трассы.
`datasets.load_sepsis` читает CSV с `keep_default_na=False`: иначе трасса с номером `NA` превращается в пропуск и теряется.

In [3]:
raw = datasets.load_sepsis(SEPSIS_PATH)
sepsis = crossval.cross_validate(raw, datasets.SEPSIS_START, datasets.SEPSIS_TERMINAL,
                                 k=K_FOLDS, seed=SEED, n_sim=N_SIM)

фолд 1/5 готов


фолд 2/5 готов


фолд 3/5 готов


фолд 4/5 готов


фолд 5/5 готов


## 2. Синтетика A–D

Истинная сеть и механизм цензуры — из `petri_net_time_estimation.ipynb` (перенесены в `survpm.datasets`).
Генератор отдаёт момент окончания наблюдения и причину цензуры каждой незавершённой трассы (`censoring`),
поэтому здесь цензура настоящая: потеря наблюдения или конец исследования.
Семейства распределений выбираются по AIC на обучающих трассах, как и для Sepsis.

In [4]:
raw_ad, censoring = datasets.generate_ad(SYNTHETIC_CASES, seed=SEED)
synthetic = crossval.cross_validate(raw_ad, datasets.AD_START, datasets.AD_TERMINAL,
                                    censoring["end"], censoring["reason"], k=K_FOLDS, seed=SEED, n_sim=N_SIM)

фолд 1/5 готов


фолд 2/5 готов


фолд 3/5 готов


фолд 4/5 готов


фолд 5/5 готов


## 3. Результаты: среднее и стандартное отклонение по фолдам

In [5]:
corpora = {"Sepsis": sepsis, "A–D": synthetic}
summaries = {}
for level in ["level1", "level2", "level3", "level4"]:
    summaries[level] = pd.concat({name: crossval.summarize(tables[level]) for name, tables in corpora.items()},
                                 names=["corpus"])
    print(level)
    display(summaries[level].round(4))

level1


fitness_token      fitness_alignments       \
                                  mean  std               mean  std   
corpus model     variant                                              
Sepsis state_net A                 1.0  0.0             1.0000  0.0   
                 Б                 1.0  0.0                NaN  NaN   
       inductive A                 1.0  0.0             0.9999  0.0   
                 Б                 1.0  0.0                NaN  NaN   
A–D    state_net A                 1.0  0.0             1.0000  0.0   
                 Б                 1.0  0.0                NaN  NaN   
       inductive A                 1.0  0.0             0.9998  0.0   
                 Б                 1.0  0.0                NaN  NaN   

                         precision_token         precision_alignments          \
                                    mean     std                 mean     std   
corpus model     variant                                                        
Sepsis state_net A                0.7750  0.0499               0.7750  0.0499   
                 Б                0.7891  0.0486                  NaN     NaN   
       inductive A                0.5966  0.0380               0.5966  0.0380   
                 Б                0.6094  0.0370                  NaN     NaN   
A–D    state_net A                0.9580  0.0084               0.9580  0.0084   
                 Б                0.9594  0.0094                  NaN     NaN   
       inductive A                0.4991  0.0064               0.4991  0.0064   
                 Б                0.5012  0.0063                  NaN     NaN   

                         generalization         simplicity       
                                   mean     std       mean  std  
corpus model     variant                                         
Sepsis state_net A               0.7358  0.0238     0.5758  0.0  
                 Б               0.7438  0.0221     0.5758  0.0  
       inductive A               0.7316  0.0014     0.6727  0.0  
                 Б               0.8491  0.0033     0.6727  0.0  
A–D    state_net A               0.8834  0.0033     0.6842  0.0  
                 Б               0.8918  0.0024     0.6842  0.0  
       inductive A               0.8007  0.0018     0.6800  0.0  
                 Б               0.8875  0.0019     0.6800  0.0

level2


entropic_relevance           uemsc          \
                                          mean     std    mean     std   
corpus model        variant                                              
Sepsis naive        A                   6.3515  0.3001  0.2276  0.0188   
                    Б                   6.5357  0.2560  0.2951  0.0147   
       censor_aware A                   6.3503  0.3019  0.2274  0.0193   
                    Б                   6.5358  0.2574  0.2948  0.0152   
A–D    naive        A                   3.7602  0.1637  0.8194  0.0404   
                    Б                   4.5202  0.1432  0.8169  0.0314   
       censor_aware A                   3.7515  0.1682  0.8277  0.0451   
                    Б                   4.5223  0.1460  0.8159  0.0304   

                                emd          
                               mean     std  
corpus model        variant                  
Sepsis naive        A        0.2547  0.0080  
                    Б        0.2215  0.0036  
       censor_aware A        0.2547  0.0073  
                    Б        0.2215  0.0044  
A–D    naive        A        0.0525  0.0182  
                    Б        0.0487  0.0094  
       censor_aware A        0.0453  0.0181  
                    Б        0.0487  0.0082

level3


temporal_deviation_share            cfld          \
                                                mean     std    mean     std   
corpus model        variant                                                    
Sepsis train_log    A                         0.2199  0.0444     NaN     NaN   
                    Б                         0.1743  0.0327     NaN     NaN   
       naive        A                         0.2736  0.0647  0.2634  0.0104   
                    Б                         0.2038  0.0485  0.2446  0.0097   
       censor_aware A                         0.2710  0.0711  0.2622  0.0110   
                    Б                         0.2019  0.0533  0.2448  0.0100   
A–D    train_log    A                         0.4199  0.0677     NaN     NaN   
                    Б                         0.3750  0.0589     NaN     NaN   
       naive        A                         0.4323  0.0802  0.0528  0.0139   
                    Б                         0.3830  0.0697  0.0669  0.0119   
       censor_aware A                         0.4099  0.0604  0.0533  0.0122   
                    Б                         0.3650  0.0451  0.0668  0.0097   

                            ngram_3         cycle_time_emd           \
                               mean     std           mean      std   
corpus model        variant                                           
Sepsis train_log    A           NaN     NaN            NaN      NaN   
                    Б           NaN     NaN            NaN      NaN   
       naive        A        0.1824  0.0059        32.1417   3.4330   
                    Б        0.1823  0.0097        22.8124   5.5330   
       censor_aware A        0.1821  0.0082        31.0283   3.5833   
                    Б        0.1852  0.0091        21.5819   2.8050   
A–D    train_log    A           NaN     NaN            NaN      NaN   
                    Б           NaN     NaN            NaN      NaN   
       naive        A        0.0483  0.0187        39.7193  12.4380   
                    Б        0.0671  0.0107        32.9370  12.9053   
       censor_aware A        0.0521  0.0179        59.0087  21.7288   
                    Б        0.0643  0.0086        49.6380  17.1062   

                            absolute_event_emd          edge_ks          \
                                          mean      std    mean     std   
corpus model        variant                                               
Sepsis train_log    A                      NaN      NaN     NaN     NaN   
                    Б                      NaN      NaN     NaN     NaN   
       naive        A                 173.2332  58.1310  0.1329  0.0085   
                    Б                 179.3121  56.6444  0.1329  0.0085   
       censor_aware A                 200.8521  55.4547  0.1344  0.0078   
                    Б                 221.6064  71.3220  0.1344  0.0078   
A–D    train_log    A                      NaN      NaN     NaN     NaN   
                    Б                      NaN      NaN     NaN     NaN   
       naive        A                  26.5326  17.1765  0.1038  0.0227   
                    Б                  23.7579  10.9674  0.1038  0.0227   
       censor_aware A                  38.1583  20.4888  0.1078  0.0243   
                    Б                  23.9462  11.0366  0.1078  0.0243   

                             km_sup         km_l1_days          \
                               mean     std       mean     std   
corpus model        variant                                      
Sepsis train_log    A           NaN     NaN        NaN     NaN   
                    Б           NaN     NaN        NaN     NaN   
       naive        A           NaN     NaN        NaN     NaN   
                    Б        0.1183  0.0160     1.1746  0.1514   
       censor_aware A           NaN     NaN        NaN     NaN   
                    Б        0.1192  0.0167     1.1705  0.1651   
A–D    train_log    A           NaN     N

level4


unseen_share      nll_per_interval         nll_per_trace  \
                            mean  std             mean     std          mean   
corpus model                                                                   
Sepsis naive                 0.0  0.0           0.1848  0.0708        0.8995   
       censor_aware          0.0  0.0           0.1845  0.0705        0.8981   
A–D    naive                 0.0  0.0           2.7425  0.0478        9.0768   
       censor_aware          0.0  0.0           2.7381  0.0434        9.0624   

                             
                        std  
corpus model                 
Sepsis naive         0.3502  
       censor_aware  0.3490  
A–D    naive         0.2937  
       censor_aware  0.2859

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for level, summary in summaries.items():
    summary.to_csv(OUTPUT_DIR / f"{level}_summary.csv")
    pd.concat({name: tables[level] for name, tables in corpora.items()}, names=["corpus", "row"]) \
        .to_csv(OUTPUT_DIR / f"{level}_folds.csv")
print("Сохранено:", OUTPUT_DIR.resolve())

Сохранено: /tmp/nbrun/dima/notebooks/output/censored_metrics


## 4. Попарные разницы по фолдам

Модели оцениваются на одних и тех же фолдах, поэтому сравнивать их честнее внутри фолда:
`difference = censor_aware − naive`. Если знак разницы одинаков во всех пяти фолдах, преимущество устойчиво,
даже когда стандартные отклонения в таблице выше перекрываются.

In [7]:
PAIRED = {"level3": ["km_sup", "km_l1_days", "cycle_time_emd", "ibs"], "level4": ["nll_per_interval"]}
for name, tables in corpora.items():
    for level, metrics in PAIRED.items():
        for metric in metrics:
            print(name, metric)
            display(crossval.paired_differences(tables[level], metric).round(4))

Sepsis km_sup


,model,censor_aware,naive,difference
fold,variant,,,
0,Б,0.1035,0.1023,0.0012
1,Б,0.1149,0.1143,0.0006
2,Б,0.1341,0.1308,0.0032
3,Б,0.1394,0.1388,0.0006
4,Б,0.1042,0.1050,-0.0008


Sepsis km_l1_days


,model,censor_aware,naive,difference
fold,variant,,,
0,Б,1.1506,1.1802,-0.0295
1,Б,0.9927,0.9852,0.0076
2,Б,1.0344,1.0638,-0.0294
3,Б,1.3744,1.3307,0.0437
4,Б,1.3004,1.3131,-0.0126


Sepsis cycle_time_emd


...

model         censor_aware    naive  difference
fold variant                                   
0    A             33.5652  30.0124      3.5528
     Б             30.7550  24.9050      5.8500
1    A             92.3913  61.0559     31.3354
     Б             61.5000  39.2300     22.2700
2    A             60.3230  33.4099     26.9130
     Б             48.1150  18.8250     29.2900
3    A             60.7950  39.8323     20.9627
     Б             71.7200  51.7550     19.9650
4    A             47.9689  34.2857     13.6832
     Б             36.1000  29.9700      6.1300

A–D ibs


,model,censor_aware,naive,difference
fold,variant,,,
0,Б,0.2104,0.2135,-0.0031
1,Б,0.2143,0.2103,0.0040
2,Б,0.2091,0.2115,-0.0024
3,Б,0.2128,0.2099,0.0030
4,Б,0.2164,0.2215,-0.0051


A–D nll_per_interval


model,censor_aware,naive,difference
fold,,,
0,2.7869,2.7972,-0.0102
1,2.6884,2.6834,0.0050
2,2.7738,2.7783,-0.0044
3,2.7007,2.7057,-0.0050
4,2.7408,2.7477,-0.0069


## 5. Ограничения

* Сеть состояний строится из всех переходов обучающих трасс, поэтому её fitness почти всегда равен 1:
  ниже он бывает только из-за переходов, которых не было в обучении. Её сравнение с Inductive Miner
  интереснее по precision и generalization.
* На Sepsis время цензуры почти всегда около нуля (момент последней записи), поэтому наивная модель и
  модель с цензурой здесь почти не различаются ни по одной метрике. Разницу показывает синтетика.
* Ebi считает entropic relevance точно, в рациональных числах, и падает с «cannot compute ER»
  при длинных знаменателях. Поэтому вероятности автомата округляются до 3 знаков (при сбое до 2).
* `temporal_deviation_share` для строки `train_log` — базовая линия: профиль построен по обучающему
  журналу, а не по модели.
* Log-rank сравнивает тестовые трассы с 5000 симулированными, поэтому при большом журнале он отвергает
  гипотезу даже при малом расхождении; смотрите в первую очередь на `km_sup` и `km_l1_days`.
